<a href="https://colab.research.google.com/github/google-ai-edge/LiteRT-CLI/blob/main/test_scripts/litert_cli.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### Copyright 2026 The LiteRT CLI Authors.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
# ==============================================================================

# LiteRT CLI Demo

This notebook demonstrates how to use the `litert-cli` tool to convert a PyTorch model, quantize it and run it.

## 🛠️ 1. Environment Setup & Installation

In [ ]:
# Update protobuf version
!pip install -U protobuf
# Install required packages
!pip install torch torchvision
# Default torchao is 
!pip install -U torchao

# Install litert-cli
!pip install litert-cli-nightly

# 'litert compile' depends on Clang, and below make sure your Clang has version `18.x.x` or above
!wget https://apt.llvm.org/llvm.sh
!chmod +x llvm.sh
!sudo ./llvm.sh 18 all

## 📝 2. Prepare PyTorch Model Script

In [ ]:
%%writefile resnet18.py
import torch
import torchvision

def get_model() -> torch.nn.Module:
  model = torchvision.models.resnet18(
      weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1
  )
  model.eval()
  return model

def get_args() -> tuple[torch.Tensor, ...]:
  return (torch.randn(1, 3, 224, 224),)

## 🔄 3. Model Conversion (PyTorch -> LiteRT)

In [ ]:
# Convert PyTorch ResNet18 model to LiteRT
!litert convert resnet18.py --output resnet18

## 📉 4. Model Quantization

In [ ]:
# Quantize the ResNet18 model
!litert quantize resnet18/resnet18.tflite --type int8_dynamic --output resnet18/resnet18_int8_dynamic.tflite
!litert quantize resnet18/resnet18.tflite --type int8_weight_only --output resnet18/resnet18_int8_weight_only.tflite

## 🚀 5. Run Inference
### 🖥️ 5.1 CPU Inference

In [ ]:
# Run Inference on Desktop (Colab VM CPU)
!litert run resnet18/resnet18.tflite --desktop --cpu --iterations 1
!litert run resnet18/resnet18_int8_dynamic.tflite --desktop --cpu --iterations 1

### 🎮 5.2 GPU Inference

In [ ]:
# Run Inference on Desktop (Colab VM GPU)
!litert run resnet18/resnet18.tflite --desktop --gpu --iterations 1
!litert run resnet18/resnet18_int8_dynamic.tflite --desktop --gpu --iterations 1

## ⚙️ 6. NPU Offline Compilation (AOT)

In [ ]:
# Compile the model for qualcomm NPU: SM8750
# TIP: Only support running on Linux and it might take a few minutes, given download large SDKs from SOCs.
# TIP: It depends on Clang, and please make sure your Clang has version `18.x.x` or above when running
#   `clang --version`. To update, choose one of below: 
#   a) `sudo apt update && sudo apt upgrade -y`
#   b) wget https://apt.llvm.org/llvm.sh
#      chmod +x llvm.sh
#      sudo ./llvm.sh 18 all
#
!litert compile resnet18/resnet18.tflite --target sm8750

## 🏁 7. Benchmark in Google AI Edge Portal

In [ ]:
# Please login into Google Cloud and make sure you have joined the EAP for Google AI Edge Portal
# Check: https://ai.google.dev/edge/ai-edge-portal
!pip install gcloud
!gcloud auth login

# Benchmark on Google AI Edge Portal
# Please specify you own GCP project: --gcp-project <your-own-gcp-project-id>
# Or set a default environment variable: LITERT_GCP_PROJECT
# Optionally specify your GCS bucket via --gcp-bucket, otherwise it will automatically create and use one.
!litert benchmark resnet18/resnet18.tflite --gcp --device "pixel 7" --cpu --gcp-project <your-gcp-project>
!litert benchmark resnet18/resnet18.tflite --gcp --devices "pixel 7, sm-s931u1" --gpu --gcp-project <your-gcp-project> --gcp-bucket <your-gcp-bucket>

## Advanced Usage

### Android Deployment
To run or benchmark on Android, you need an ADB connected device. These commands typically look like:
```bash
# Run on Android
!litert run resnet18/resnet18.tflite --android --cpu

# Benchmark on Android
!litert benchmark resnet18/resnet18.tflite --android
```
These are not executed in this notebook as Colab does not have access to a physical Android device by default.